# 06 — IndoBERT vs XLM-RoBERTa

Notebook ini memakai pasangan **kalimat dan nama aspek** dari split tahap 05. Kedua model menggunakan empat kombinasi tuning, tiga epoch maksimum, dan lima seed. Pemilihan parameter dan checkpoint menggunakan validasi; test hanya dievaluasi setelah pelatihan kedua model.

Pilih kernel Conda `DeepLearning` dan jalankan dari folder `repo_bank_news`. Bobot dan checkpoint memerlukan ruang penyimpanan beberapa GB. Hasil yang sudah ada dapat ditampilkan tanpa melatih ulang.

## 1. Konfigurasi

`OUTPUT_DIR = "results"` menggunakan hasil yang telah selesai. Untuk eksperimen baru yang tidak memakai hasil tersimpan, ganti ke nama folder baru, misalnya `results_baru`. Set `DOWNLOAD_MODELS` atau `TRAIN_MODELS` menjadi `True` hanya jika ingin menjalankan tahap tersebut.

In [ ]:
OUTPUT_DIR = "results"
DOWNLOAD_MODELS = False
TRAIN_MODELS = False

## 2. Unduh bobot resmi bila belum tersedia

Unduhan memverifikasi SHA-256. Biarkan dinonaktifkan jika `model_assets` sudah tersedia.

In [ ]:
if DOWNLOAD_MODELS:
    """Download official pretrained weights and tokenizers into model_assets/."""
    import os
    os.environ['USE_TF'] = '0'
    from pathlib import Path
    from concurrent.futures import ThreadPoolExecutor
    import hashlib
    import math
    import requests
    from transformers import AutoConfig, AutoTokenizer
    
    ROOT = Path.cwd()
    MODELS = {'indobert': ('indobenchmark/indobert-base-p1', 'pytorch_model.bin'),
              'xlmr': ('FacebookAI/xlm-roberta-base', 'model.safetensors')}
    for name, (repo, filename) in MODELS.items():
        out = ROOT/'model_assets'/name
        out.mkdir(parents=True, exist_ok=True)
        AutoConfig.from_pretrained(repo).save_pretrained(out)
        AutoTokenizer.from_pretrained(repo).save_pretrained(out)
        path = out/filename
        meta = requests.get(f'https://huggingface.co/api/models/{repo}?blobs=true',timeout=30)
        meta.raise_for_status()
        entry = next(x for x in meta.json()['siblings'] if x['rfilename']==filename)
        expected = entry['size']; digest = entry['lfs']['sha256']
        if path.exists():
            sha=hashlib.sha256()
            with path.open('rb') as handle:
                while chunk := handle.read(4*1024*1024): sha.update(chunk)
            if path.stat().st_size != expected or sha.hexdigest() != digest: raise RuntimeError(f'Invalid existing file: {path}')
            print(f'{name}: existing weights SHA-256 verified', flush=True)
            continue
        tmp = path.with_suffix(path.suffix+'.part')
        start = tmp.stat().st_size if tmp.exists() else 0
        if start > expected: raise RuntimeError('Partial file is larger than expected')
        width = math.ceil((expected-start)/8)
        ranges = [(i, min(i+width-1,expected-1)) for i in range(start,expected,width)] if width else []
        def download(bounds):
            first,last=bounds; segment=out/f'{filename}.segment-{first}-{last}'
            if segment.exists() and segment.stat().st_size==last-first+1: return segment
            with requests.get(f'https://huggingface.co/{repo}/resolve/main/{filename}', headers={'Range':f'bytes={first}-{last}'}, stream=True,timeout=(30,120)) as response:
                response.raise_for_status()
                if response.status_code != 206 or not response.headers.get('Content-Range','').startswith(f'bytes {first}-{last}/'):
                    raise RuntimeError(f'Invalid range response: {response.status_code} {response.headers.get("Content-Range")}')
                with segment.open('wb') as handle:
                    for chunk in response.iter_content(1024*1024):
                        if chunk: handle.write(chunk)
            if segment.stat().st_size!=last-first+1: raise RuntimeError(f'Incomplete segment: {segment}')
            print(f'{name}: segment {first}-{last} selesai',flush=True)
            return segment
        with ThreadPoolExecutor(max_workers=8) as pool: segments=list(pool.map(download,ranges))
        with tmp.open('ab') as output:
            for segment in segments:
                with segment.open('rb') as source:
                    while data := source.read(4*1024*1024): output.write(data)
                segment.unlink()
        if tmp.stat().st_size != expected: raise RuntimeError('Incomplete assembled model')
        sha=hashlib.sha256()
        with tmp.open('rb') as handle:
            while chunk := handle.read(4*1024*1024): sha.update(chunk)
        if sha.hexdigest()!=digest: raise RuntimeError('Model SHA-256 mismatch; do not use file')
        tmp.replace(path)
        print(f'{name}: downloaded and SHA-256 verified ({expected} bytes)', flush=True)


## 3. Persiapan data dan konfigurasi eksperimen

In [ ]:
"""Fine-tune sentence/aspect pairs. Select only on validation macro F1."""
import gc
import json
import os
import random
from pathlib import Path
os.environ['USE_TF'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

ROOT = Path.cwd()
OUT = ROOT / OUTPUT_DIR; OUT.mkdir(exist_ok=True)
MODELS = {'indobert':str(ROOT/'model_assets/indobert'), 'xlmr':str(ROOT/'model_assets/xlmr')}
LABELS = ['negatif', 'netral', 'positif']
TRIALS = [{'lr':lr, 'weight_decay':wd} for lr in [2e-5, 3e-5] for wd in [0.0, 0.01]]
SEEDS = [42, 43, 44, 45, 46]
EPOCHS = 3
MAX_LENGTH = 192
BATCH_SIZE = 8
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda': raise RuntimeError('GPU CUDA diperlukan untuk eksperimen ini.')
frames = {s:pd.read_csv(ROOT/f'data/final/{s}.csv', dtype={'id_artikel':str}) for s in ['train','val','test']}



## 4. Dataset, training, dan evaluasi

In [ ]:
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

class Rows(Dataset):
    def __init__(self, frame, tokenizer):
        self.enc = tokenizer(frame.kalimat.tolist(), frame.nama_aspek.tolist(), truncation=True, max_length=MAX_LENGTH)
        self.labels = frame.sentimen.to_numpy()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return {**{k:v[i] for k,v in self.enc.items()}, 'labels':int(self.labels[i])}

def collate(batch, tokenizer):
    labels = torch.tensor([r.pop('labels') for r in batch])
    padded = tokenizer.pad(batch, padding=True, return_tensors='pt')
    padded['labels'] = labels
    return padded

def predict(model, loader):
    model.eval(); labels=[]; predictions=[]; probs=[]
    with torch.no_grad():
        for batch in loader:
            gold=batch.pop('labels').numpy(); batch={k:v.to(device) for k,v in batch.items()}
            with torch.autocast(device_type='cuda', dtype=torch.float16): logits=model(**batch).logits
            p=torch.softmax(logits.float(), dim=-1).cpu().numpy()
            labels.extend(gold.tolist()); predictions.extend(p.argmax(1).tolist()); probs.extend(p.tolist())
    return np.array(labels), np.array(predictions), np.array(probs)

def metrics(y, p):
    pre, rec, f1, support=precision_recall_fscore_support(y,p,labels=[0,1,2],zero_division=0)
    return {'accuracy':accuracy_score(y,p), 'macro_precision':float(pre.mean()), 'macro_recall':float(rec.mean()), 'macro_f1':float(f1.mean()), 'per_class':{name:{'precision':float(pre[i]),'recall':float(rec[i]),'f1':float(f1[i]),'support':int(support[i])} for i,name in enumerate(LABELS)}, 'confusion_matrix':confusion_matrix(y,p,labels=[0,1,2]).tolist()}

def fit(name, params, seed, tag, final=False):
    run=OUT/name/tag; result=run/'result.json'
    if result.exists(): return json.loads(result.read_text(encoding='utf-8'))
    run.mkdir(parents=True, exist_ok=True); seed_all(seed)
    tokenizer=AutoTokenizer.from_pretrained(MODELS[name]); sets={s:Rows(f,tokenizer) for s,f in frames.items() if s != 'test'}
    loaders={s:DataLoader(ds,batch_size=BATCH_SIZE,shuffle=(s=='train'),collate_fn=lambda batch:collate(batch,tokenizer)) for s,ds in sets.items()}
    model=AutoModelForSequenceClassification.from_pretrained(MODELS[name], num_labels=3).to(device)
    optim=torch.optim.AdamW(model.parameters(),lr=params['lr'],weight_decay=params['weight_decay'])
    steps=len(loaders['train'])*EPOCHS
    scheduler=get_linear_schedule_with_warmup(optim,num_warmup_steps=int(steps*.1),num_training_steps=steps)
    scaler=torch.amp.GradScaler('cuda'); best=-1; best_epoch=0
    for epoch in range(1,EPOCHS+1):
        model.train(); total=0
        for batch in loaders['train']:
            batch={k:v.to(device) for k,v in batch.items()}; optim.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16): loss=model(**batch).loss
            scaler.scale(loss).backward(); scaler.unscale_(optim); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optim); scaler.update(); scheduler.step(); total+=loss.item()
        y,p,_=predict(model,loaders['val']); score=metrics(y,p)['macro_f1']
        print(f'{name} {tag} epoch={epoch} loss={total/len(loaders["train"]):.4f} val_macro_f1={score:.4f}', flush=True)
        if score>best:
            best=score; best_epoch=epoch
            torch.save(model.state_dict(),run/'best.pt')
    result_data={'model':name,'params':params,'seed':seed,'best_epoch':best_epoch,'validation_macro_f1':best}
    result.write_text(json.dumps(result_data,indent=2,ensure_ascii=False),encoding='utf-8')
    if not final: (run/'best.pt').unlink()
    del model, optim, scheduler, loaders, sets; gc.collect(); torch.cuda.empty_cache()
    return result_data

def evaluate(name, seed):
    run=OUT/name/f'seed_{seed}'; result=run/'result.json'
    data=json.loads(result.read_text(encoding='utf-8'))
    if 'test' in data: return data
    tokenizer=AutoTokenizer.from_pretrained(MODELS[name])
    loader=DataLoader(Rows(frames['test'],tokenizer),batch_size=BATCH_SIZE,shuffle=False,collate_fn=lambda batch:collate(batch,tokenizer))
    model=AutoModelForSequenceClassification.from_pretrained(MODELS[name],num_labels=3).to(device)
    model.load_state_dict(torch.load(run/'best.pt',map_location=device,weights_only=True))
    y,p,probs=predict(model,loader); data['test']=metrics(y,p)
    pred=frames['test'].copy(); pred['prediksi']=p; pred['label_aktual']=pred.sentimen.map(dict(enumerate(LABELS))); pred['label_prediksi']=pred.prediksi.map(dict(enumerate(LABELS)))
    for i,label in enumerate(LABELS): pred['prob_'+label]=probs[:,i]
    pred.to_csv(run/'predictions.csv',index=False)
    data['per_aspect']={str(a):metrics(g.sentimen.to_numpy(),g.prediksi.to_numpy()) for a,g in pred.groupby('nama_aspek')}
    result.write_text(json.dumps(data,indent=2,ensure_ascii=False),encoding='utf-8')
    del model,loader; gc.collect(); torch.cuda.empty_cache()
    return data



## 5. Tuning dan pelatihan lima seed

Sel ini tidak berjalan secara default. Run yang sudah lengkap di folder output akan digunakan kembali; pilih folder output baru untuk menjalankan ulang semuanya.

In [ ]:
def main():
    # Fit final seeds as each model becomes available; test only after both are tuned.
    chosen={}
    for name in MODELS:
        tuning=[fit(name,p,42,f'tuning_{i}') for i,p in enumerate(TRIALS)]
        chosen[name]=max(tuning,key=lambda x:x['validation_macro_f1'])['params']
        for seed in SEEDS: fit(name,chosen[name],seed,f'seed_{seed}',final=True)
    (OUT/'selected_hyperparameters.json').write_text(json.dumps(chosen,indent=2),encoding='utf-8')
    results={name:[evaluate(name,seed) for seed in SEEDS] for name in MODELS}
    summary={name:{metric:{'mean':float(np.mean([x['test'][metric] for x in runs])),'std':float(np.std([x['test'][metric] for x in runs],ddof=1))} for metric in ['macro_f1','accuracy','macro_precision','macro_recall']} for name,runs in results.items()}
    (OUT/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    # Paired article-cluster bootstrap for seed 42: preserve within-article dependence.
    a=pd.read_csv(OUT/'indobert/seed_42/predictions.csv',dtype={'id_artikel':str})
    b=pd.read_csv(OUT/'xlmr/seed_42/predictions.csv',dtype={'id_artikel':str})
    assert a.id_kalimat.equals(b.id_kalimat) and a.sentimen.equals(b.sentimen)
    article_rows=list(a.groupby('id_artikel').indices.values()); rng=np.random.default_rng(42); delta=[]
    for _ in range(2000):
        selected=rng.integers(0,len(article_rows),len(article_rows))
        idx=np.concatenate([article_rows[i] for i in selected])
        y=a.sentimen.to_numpy()[idx]
        delta.append(metrics(y,a.prediksi.to_numpy()[idx])['macro_f1']-metrics(y,b.prediksi.to_numpy()[idx])['macro_f1'])
    comparison={'description':'Perbedaan macro-F1 IndoBERT minus XLM-R, paired bootstrap berdasarkan artikel, seed 42 saja', 'bootstrap_replicates':2000, 'ci95':np.percentile(delta,[2.5,97.5]).tolist()}
    (OUT/'bootstrap_seed_42.json').write_text(json.dumps(comparison,indent=2,ensure_ascii=False),encoding='utf-8')
    print(json.dumps(summary,indent=2),flush=True)


if TRAIN_MODELS:
    main()

## 6. Tampilkan hasil test tersimpan

In [ ]:
import json
import pandas as pd
from IPython.display import display
summary_path = OUT / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    table = pd.DataFrame({model: {metric: f"{value['mean']:.4f} ± {value['std']:.4f}" for metric, value in metrics.items()} for model, metrics in summary.items()}).T
    display(table)
else:
    print('Belum ada hasil test. Aktifkan TRAIN_MODELS untuk menjalankan eksperimen.')

## 7. Prediksi dan metrik per kelas

Pilih model dan seed untuk memeriksa prediksi per baris serta precision, recall, F1, support dan confusion matrix. Metrik per aspek tersedia dalam `result["per_aspect"]`.

In [ ]:
MODEL = 'indobert'  # 'indobert' atau 'xlmr'
SEED = 42
run = OUT / MODEL / f'seed_{SEED}'
if (run / 'predictions.csv').exists():
    predictions = pd.read_csv(run / 'predictions.csv')
    result = json.loads((run / 'result.json').read_text(encoding='utf-8'))
    display(predictions[['id_kalimat', 'kalimat', 'nama_aspek', 'label_aktual', 'label_prediksi', 'prob_negatif', 'prob_netral', 'prob_positif']].head(20))
    display(pd.DataFrame(result['test']['per_class']).T)
    display(pd.DataFrame(result['test']['confusion_matrix'], index=['aktual_negatif','aktual_netral','aktual_positif'], columns=['pred_negatif','pred_netral','pred_positif']))
else:
    print('Prediksi untuk model dan seed ini belum tersedia.')

## Batasan

Bootstrap berpasangan berdasarkan artikel tersedia di `results/bootstrap_seed_42.json` dan hanya untuk seed 42. Provenance anotasi dan penyelesaian konflik perlu diverifikasi. Kappa, baseline, dan ablasi proposal belum dikerjakan.